In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import os


In [2]:
# Step 1: Load Data
def load_data(ticker, start_date="2000-01-01", end_date=None):
    try:
        if end_date:  # If an end date is provided (for historical prediction)
            data = yf.download(ticker, start=start_date, end=end_date)
        else:  # If no end date is provided (for future prediction)
            data = yf.download(ticker, start=start_date)  # Start from a default date (e.g., 2000-01-01)
        if data.empty:
            raise ValueError(f"Failed to retrieve data for {ticker}. Please check the ticker symbol or try again later.")
        os.makedirs('data', exist_ok=True)
        data.to_csv('data/stock_data.csv')
        return data
    except Exception as e:
        print(f"Error occurred while fetching data: {e}")
        return None


In [3]:
# Step 2: Preprocess Data
def preprocess_data(data):
    data = data[['Close']]
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)
    return scaled_data, scaler

In [4]:
# Step 3: Prepare Training and Testing Datasets
def create_datasets(data, time_step=60):
    x_data, y_data = [], []
    for i in range(len(data) - time_step):
        x_data.append(data[i:(i + time_step), 0])
        y_data.append(data[i + time_step, 0])
    return np.array(x_data), np.array(y_data)

In [5]:
# Step 4: Build LSTM Model
def build_model():
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(60, 1)),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(25),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model


In [6]:
# Step 5: Train and Evaluate Model
def train_model(model, x_train, y_train, epochs=50, batch_size=32):
    model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, verbose=1)
    return model


In [7]:
# Step 6: Predict and Visualize Results
def predict_and_plot(model, scaler, x_test, y_test, original_data):
    predictions = model.predict(x_test)
    predictions = scaler.inverse_transform(predictions)
    y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

    plt.figure(figsize=(12, 6))
    plt.plot(original_data.index[-len(y_test):], y_test, color='blue', label='Actual Price')
    plt.plot(original_data.index[-len(y_test):], predictions, color='red', label='Predicted Price')
    plt.title('Stock Price Prediction')
    plt.xlabel('Date')
    plt.ylabel('Stock Price')
    plt.legend()
    plt.show()
    

In [8]:
# Define predict_future function for future predictions
def predict_future(model, scaler, scaled_data, time_step, future_days):
    # Start with the last sequence of data available
    last_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)

    future_predictions = []

    for _ in range(future_days):
        # Make a prediction for the next day
        predicted_price = model.predict(last_sequence)
        
        # Inverse transform the prediction back to original scale
        predicted_price = scaler.inverse_transform(predicted_price)

        future_predictions.append(predicted_price[0, 0])

        # Update the sequence to include the new prediction
        last_sequence = np.append(last_sequence[:, 1:, :], predicted_price.reshape(1, 1, 1), axis=1)

    return future_predictions


In [17]:
# Main function for the program
def main():
    print("Choose an option:")
    print("1. Predict Known Data (Historical Prediction)")
    print("2. Predict Future Data")
    choice = input("Enter 1 for Historical Prediction or 2 for Future Prediction: ")

    if choice == "1":
        ticker = input("Enter the stock ticker (e.g., AAPL for Apple Inc.): ")
        start_date = input("Enter the starting date (YYYY-MM-DD): ")
        end_date = input("Enter the ending date (YYYY-MM-DD): ")
        time_step = 60

        # Load and preprocess data
        data = load_data(ticker, start_date, end_date)
        if data is None:
            return  # Exit the program if data could not be fetched

        print("\nInput Data (Historical):")
        print(data.head())  # Display only the first few rows

        scaled_data, scaler = preprocess_data(data)

        # Create datasets
        x, y = create_datasets(scaled_data, time_step)

        # Check if x has enough data to reshape
        if x.shape[0] == 0:
            print("Not enough data to create sequences.")
            return

        # Check for missing values in x and y
        if np.any(np.isnan(x)) or np.any(np.isnan(y)):
            print("Data contains NaN values. Please check the dataset.")
            return

        # Check for negative values (in case of scale issues)
        if np.any(x < 0) or np.any(y < 0):
            print("Data contains negative values. Please check the dataset.")
            return

        x = x.reshape((x.shape[0], x.shape[1], 1))  # Reshape for LSTM

        # Split into train and test
        split = int(len(x) * 0.8)
        x_train, x_test = x[:split], x[split:]
        y_train, y_test = y[:split], y[split:]

        # Build, train, and evaluate model
        model = build_model()
        model = train_model(model, x_train, y_train)

        # Predict and visualize
        predictions = model.predict(x_test)
        predictions = scaler.inverse_transform(predictions)
        y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

        # Plot the results for Historical Prediction
        plt.figure(figsize=(12, 6))
        plt.plot(data.index[-len(y_test):], y_test, color='blue', label='Actual Price')
        plt.plot(data.index[-len(y_test):], predictions, color='red', label='Predicted Price')
        plt.title(f'{ticker} Stock Price Prediction (Historical Data)')
        plt.xlabel('Date')
        plt.ylabel('Stock Price')
        plt.legend()
        plt.grid(True)
        plt.show()

    elif choice == "2":
        ticker = input("Enter the stock ticker (e.g., AAPL for Apple Inc.): ")
        future_days = int(input("Enter the number of days to predict into the future: "))
        start_date = input("Enter the starting date for prediction (YYYY-MM-DD): ")  # User input for starting date
        time_step = 60

        # Load and preprocess data
        data = load_data(ticker)  # No need for start_date, end_date for future prediction
        if data is None:
            return  # Exit the program if data could not be fetched

        print("\nInput Data (Historical for Future Prediction):")
        print(data.head())  # Display only the first few rows

        scaled_data, scaler = preprocess_data(data)

        # Create datasets for training
        x, y = create_datasets(scaled_data, time_step)
        if x.shape[0] == 0:
            print("Not enough data to create sequences.")
            return

        # Check for missing values in x and y
        if np.any(np.isnan(x)) or np.any(np.isnan(y)):
            print("Data contains NaN values. Please check the dataset.")
            return

        # Check for negative values (in case of scale issues)
        if np.any(x < 0) or np.any(y < 0):
            print("Data contains negative values. Please check the dataset.")
            return

        x = x.reshape((x.shape[0], x.shape[1], 1))  # Reshape for LSTM

        # Split into train and test
        split = int(len(x) * 0.8)
        x_train, x_test = x[:split], x[split:]
        y_train, y_test = y[:split], y[split:]

        # Build and train model
        model = build_model()
        model = train_model(model, x_train, y_train)

        # Predict future stock prices
        future_predictions = predict_future(model, scaler, scaled_data, time_step, future_days)

        # Use the user-specified start date for future predictions
        future_dates = pd.date_range(start=start_date, periods=future_days, freq='B')  # Starting from user input
        future_df = pd.DataFrame(future_predictions, index=future_dates, columns=['Predicted Price'])

        # Show only predictions from the user-specified start date
        print(f"\nFuture {future_days} days predictions for {ticker} starting from {start_date}:")
        print(future_df)

        # Plot future predictions starting from the user-specified date
        plt.figure(figsize=(12, 6))
        plt.plot(future_df.index, future_df['Predicted Price'], color='orange', label=f'Predicted Price from {start_date}')
        plt.title(f'{ticker} Future Stock Price Predictions')
        plt.xlabel('Date')
        plt.ylabel('Predicted Stock Price')
        plt.legend()
        plt.grid(True)
        plt.show()

    else:
        print("Invalid input. Please choose 1 or 2.")

# Run the program
if __name__ == '__main__':
    main()

Choose an option:
1. Predict Known Data (Historical Prediction)
2. Predict Future Data


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'query2.finance.yahoo.com\', port=443): Max retries exceeded with url: /v8/finance/chart/%ticker%?period1=1675227600&period2=1706763600&interval=1d&includePrePost=False&events=div%2Csplits%2CcapitalGains&crumb=JBnJQtfHD5V (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002BB86007130>: Failed to resolve \'query2.finance.yahoo.com\' ([Errno 11001] getaddrinfo failed)"))'))


Error occurred while fetching data: Failed to retrieve data for aapl. Please check the ticker symbol or try again later.
